# Predictions using locally deployed specialized models

Two of the specialized NMR structure-elucidation models — **NMRMind** and **NMRPeak** — were deployed and run **locally** on our workstation (AMD EPYC 9654 CPU, 256 GB RAM, NVIDIA RTX A6000 48 GB); inference ran on the GPU. Both consume the experimental 1H and 13C peak lists from `dataset_selected_clean_105.json` (this notebook sits next to that file in `dataset/`).

For each model this notebook documents:
1. **Preprocessing** — how the raw NMR strings are converted into the model's input representation.
2. **Run** — the exact command and model weights used to generate the ranked SMILES predictions on the GPU workstation.

The other two specialized tools, **BLIND** and **NMR-Solver**, were used as online services and are therefore not covered here.

## 1. NMRMind

NMRMind represents each spectrum as a sequence of discrete **shift tokens**. The raw dataset strings are converted with the tokenization script `nmr2tok.py`:

- each **1H** signal → a token `H_<shift>` (two decimals), repeated once per proton (its integral); a signal given as a range is collapsed to the range midpoint;
- each **13C** signal → a token `C_<shift>` (one decimal), repeated per equivalent-carbon count;
- shifts are clamped to the model ranges (0–15 ppm for 1H, 0–230 ppm for 13C).

The converted spectra are written as one JSON record per molecule (fields `1H_NMR`, `13C_NMR`, `smiles`) into `dataset_for_nmrmind_testing.json` — the file passed to NMRMind at inference (see the run cell below). Source repository: https://github.com/WJmodels/NMRMind

In [ ]:
import re, json

# NMRMind shift-token ranges / precision (from the model's dataset loader).
H_MIN, H_MAX = 0.00, 15.00   # 1H: two decimals
C_MIN, C_MAX = 0.0, 230.0    # 13C: one decimal


def split_signals(s):
    """Split a peak list on top-level commas, ignoring commas inside parentheses."""
    parts, depth, cur = [], 0, ""
    for ch in s:
        if ch == "(":
            depth += 1; cur += ch
        elif ch == ")":
            depth = max(0, depth - 1); cur += ch
        elif ch == "," and depth == 0:
            parts.append(cur); cur = ""
        else:
            cur += ch
    if cur.strip():
        parts.append(cur)
    return [p.strip() for p in parts if p.strip()]


def strip_header(s):
    """Drop the '...NMR (...)' header, keeping only the peak list after the delta symbol."""
    for delim in ("\u03b4", "d ", ") "):
        if delim in s:
            return s.split(delim, 1)[1].strip()
    return s.strip()


def tokenize_1h(s):
    """1H string -> ['H_x.xx', ...], each shift repeated once per proton (its integral)."""
    tokens = []
    for chunk in split_signals(strip_header(s)):
        rng = re.match(r"\s*(\d+\.?\d*)\s*[-\u2013\u2014]\s*(\d+\.?\d*)", chunk)
        one = re.match(r"\s*(\d+\.?\d*)", chunk)
        if rng:
            shift = (float(rng.group(1)) + float(rng.group(2))) / 2.0   # range -> midpoint
        elif one:
            shift = float(one.group(1))
        else:
            continue
        n_h = re.findall(r"(\d+)\s*H(?![a-z])", chunk)                 # integral, not 'Hz'
        n = int(n_h[-1]) if n_h else 1
        shift = min(max(shift, H_MIN), H_MAX)
        tokens += ["H_{:.2f}".format(shift)] * n
    return tokens


def tokenize_13c(s):
    """13C string -> ['C_x.x', ...], each shift repeated per equivalent-carbon count."""
    tokens = []
    for chunk in split_signals(strip_header(s)):
        m = re.search(r"(\d+\.?\d*)", chunk)
        if not m:
            continue
        shift = min(max(float(m.group(1)), C_MIN), C_MAX)
        n_c = re.search(r"(\d+)\s*C(?![a-z])", chunk)                  # e.g. '(2C)'
        n = int(n_c.group(1)) if n_c else 1
        tokens += ["C_{:.1f}".format(shift)] * n
    return tokens


with open("dataset_selected_clean_105.json", encoding="utf-8") as f:
    dataset = json.load(f)

# One JSON record per molecule; keys match NMRMind's data loader (smiles is a 1-element list).
with open("dataset_for_nmrmind_testing.json", "w", encoding="utf-8") as out:
    for cls_name, entries in dataset.items():
        for key, rec in entries.items():
            out.write(json.dumps({
                "smiles":  [rec["smiles"]],
                "1H_NMR":  tokenize_1h(rec["h_nmr"]),
                "13C_NMR": tokenize_13c(rec["c_nmr"]),
                "id":      f"{cls_name}/{key}",
            }, ensure_ascii=False) + "\n")

# quick look at the first molecule
ex = dataset["cls_alkanes_haloalkanes"]["1.0"]
print("1H :", tokenize_1h(ex["h_nmr"]))
print("13C:", tokenize_13c(ex["c_nmr"]))

In [ ]:
%%bash
# NMRMind - forward (spectrum -> structure) inference on the GPU workstation, run from the
# NMRMind repository root. Approximate command; exact paths are machine-dependent.
# Spectrum-only input (1H + 13C). Model weights: weight/epoch_200_loss_0.061932.pth
MASTER_PORT=$(shuf -n 1 -i 10000-65535)

torchrun \
    --nproc_per_node=1 --nnodes=1 --node_rank 0 --master_addr localhost --master_port $MASTER_PORT \
    main.py \
    --max_length=2048 \
    --output_dir=weight/weight_01 \
    --gpus=1 \
    --num_train_epochs=200 \
    --num_workers=1 \
    --batch_size=1 \
    --learning_rate=0.00005 \
    --weight_decay=0.0001 \
    --num_warmup_epochs=1 \
    --use_molecular_formula_prob=1.0 \
    --use_1H_NMR_prob=1.0 \
    --use_13C_NMR_prob=1.0 \
    --use_COSY_prob=1.0 \
    --use_HMBC_prob=1.0 \
    --use_HH_prob=1.0 \
    --use_smiles_prob=1.0 \
    --config_json_path=configs/bart.json \
    --seed=42 \
    --do_test \
    --input_name=1H_NMR,13C_NMR,COSY,HMBC,HH,molecular_formula \
    --output_name=smiles \
    --mode=forward \
    --num_beams=10 \
    --tokenizer_path=tokenizer \
    --test_folder=dataset_for_nmrmind_testing.json \
    --output_dir=result \
    --model_weight=weight/epoch_200_loss_0.061932.pth \

## 2. NMRPeak

NMRPeak consumes **structured peak objects**, so the raw 1H/13C strings are parsed into:

- each **1H** signal → `{category, j_values, nH, rangeMax, rangeMin}`, where `category` is the multiplicity abbreviation normalized to the model vocabulary (unknown labels fall back to `m`);
- each **13C** signal → `{"delta (ppm)": <shift>}`, with coupling annotations in parentheses (e.g. C–F splitting) stripped first.

The model is run in `CH` mode (1H + 13C spectra only) with a beam size of 10. Source repository: https://github.com/Colin-Jay/NMRPeak

In [ ]:
import re, json


def _dash(s):
    return s.replace("\u2013", "-").replace("\u2014", "-").replace("\u2212", "-")


def norm_category(cat):
    """Normalize a journal multiplicity label to the model's vocabulary (unknown -> 'm')."""
    c = cat.strip().lower()
    c = c.replace("quint", "p").replace("pent", "p").replace("sext", "hex")
    c = c.replace("sept", "hept").replace("sep", "hept")
    if c.startswith("app"):
        rest = c[3:].strip(" ._"); c = "app_" + rest if rest else "app_s"
    elif c.startswith("br") and len(c) > 2:
        c = "br" + c[2:].strip(" ._")
    return c.replace(" ", "") or "m"


def parse_1h(s):
    """1H string -> [{category, j_values, nH, rangeMax, rangeMin}, ...]."""
    s = _dash(s)
    s = s.split("\u03b4", 1)[1] if "\u03b4" in s else re.sub(r"^\s*\d?\s*H?\s*NMR\s*\([^)]*\)", "", s)
    peaks = []
    for m in re.finditer(r"(\d+\.?\d*\s*-\s*\d+\.?\d*|\d+\.?\d*)\s*\(([^)]*)\)", s):
        shift, content = m.group(1), m.group(2)
        if "-" in shift:
            a, b = [float(x) for x in shift.split("-")]; rmax, rmin = max(a, b), min(a, b)
        else:
            rmax = rmin = float(shift)
        parts = [p.strip() for p in content.split(",")]
        jm = re.search(r"J\s*=\s*([0-9.,\s]+)", content)
        jvals = [x for x in re.split(r"[,\s]+", jm.group(1).strip()) if x] if jm else []
        nhm = re.search(r"(\d+)\s*H\b", content)
        peaks.append({
            "category": norm_category(parts[0]) if parts else "m",
            "j_values": ("_".join(jvals) + "_") if jvals else "_",
            "nH": int(nhm.group(1)) if nhm else 1,
            "rangeMax": rmax, "rangeMin": rmin,
        })
    return peaks


def parse_13c(s):
    """13C string -> [{'delta (ppm)': float}, ...] (coupling annotations dropped)."""
    s = _dash(s)
    s = s.split("\u03b4", 1)[1] if "\u03b4" in s else re.sub(r"^\s*\d{0,2}\s*C?\s*NMR\s*\([^)]*\)", "", s)
    s = re.sub(r"\([^)]*\)", " ", s)
    return [{"delta (ppm)": float(n)} for n in re.findall(r"-?\d+\.?\d*", s)]


with open("dataset_selected_clean_105.json", encoding="utf-8") as f:
    dataset = json.load(f)

nmrpeak_inputs = {
    cls_name: {
        key: {
            "h_nmr_peaks": parse_1h(rec["h_nmr"]),
            "c_nmr_peaks": parse_13c(rec["c_nmr"]),
        }
        for key, rec in entries.items()
    }
    for cls_name, entries in dataset.items()
}

ex = dataset["cls_alkanes_haloalkanes"]["1.0"]
print("1H :", parse_1h(ex["h_nmr"]))
print("13C:", parse_13c(ex["c_nmr"]))

In [ ]:
# NMRPeak prediction, as run on the GPU workstation. Illustrative: it needs the NMRPeak
# package and its trained weights, and the paths below are the workstation's (machine-dependent).
# Spectrum-only, CH mode (1H + 13C); consumes the `nmrpeak_inputs` built in the cell above.
import json
from nmrpeak.api.nmrpeak_generation import get_generation_config, api_nmrpeak_generation
from nmrpeak.api.nmrpeak_prediction import get_prediction_config
from nmrpeak.api.nmrpeak_retrieval import get_retrieval_config

DICT_PATH = "/mnt/hard1/NMRPeak/dict"
BASE_DIR  = "/mnt/hard1/NMRPeak/weights"
GENERATION_WEIGHT_DIR = "NMRexp_lr3e-4_bs16_gpu8_spec_trans_mol_bart_base_spec_trans_mol_60000_1000000"
PREDICTION_WEIGHT_DIR = "NMRexp_lr3e-4_bs16_gpu8_unimol_pretrain1_frozen0_mol_trans_spec_bart_base_mol_trans_spec_60000_1000000_best"
RETRIEVAL_WEIGHT_DIR  = "NMRexp_lr3e-4_bs512_gpu1_t0.1_unimol_pretrain1_frozen0_spec_mol_cl_bart_base_spec_mol_cl_15000_250000"

# load the three checkpoints once: generation + two rerankers (prediction, retrieval)
generation_config = get_generation_config(
    dict_path=DICT_PATH, base_dir=BASE_DIR, generation_weight_dir=GENERATION_WEIGHT_DIR)
prediction_config = get_prediction_config(
    dict_path=DICT_PATH, base_dir=BASE_DIR, prediction_weight_dir=PREDICTION_WEIGHT_DIR)
retrieval_config = get_retrieval_config(
    dict_path=DICT_PATH, base_dir=BASE_DIR, retrieval_weight_dir=RETRIEVAL_WEIGHT_DIR, batch_size=64)

# CH rerank weighting (spectra only, no molecular formula)
weighting = dict(spec_to_mol_emb_weight=0.0, spec_to_spec_emb_weight=0.0,
                 spec_to_spec_rule_weight=1.0, spec_to_spec_rule_c_weight=0.7)

# one compound at a time: feed its parsed 1H + 13C peaks, keep the ranked SMILES + scores
results = {}
for cls_name, entries in nmrpeak_inputs.items():
    results[cls_name] = {}
    for key, spec in entries.items():
        input_info = {"c_nmr_peaks": spec["c_nmr_peaks"], "h_nmr_peaks": spec["h_nmr_peaks"]}
        model_result, model_scores = api_nmrpeak_generation(
            spec_list=[input_info], nmr_type="CH",
            generation_config=generation_config, beam_size=10,
            prediction_config=prediction_config, rerank=True,
            retrieval_config=retrieval_config, use_nmrpeak_retrieval=True,
            use_valid_filter=True, use_unique_filter=True, use_formula_filter=False,
            **weighting,
        )
        results[cls_name][key] = [
            {"rank": i + 1, "smiles": smi, "score": float(sc)}
            for i, (smi, sc) in enumerate(zip(model_result[0], model_scores[0]))
        ]

with open("nmrpeak_results_ch.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)